In [1]:
import pyomo.environ as pyo
import math
from pyDOE2 import fullfact
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
model = pyo.ConcreteModel()

# define the model parameters
rho = 50
a1 = 5.9755 * 10**9
a2 = 2.5962 * 10**12
a3 = 9.6283 * 10**15
reaction_number = [1, 2, 3]
model.a = pyo.Param(reaction_number, initialize={1:a1, 2:a2, 3:a3})

# define the model variables
# decision variables
model.V = pyo.Var(bounds=(0.03, 0.1), initialize=0.06)
model.T = pyo.Var(bounds=(5.8, 6.8), initialize=5.8)
model.T2 = pyo.Var(bounds=(0.147, 1.725), initialize=0.5)
model.FA = pyo.Var(bounds=(1, None), initialize=10)
model.FB = pyo.Var(bounds=(1, None), initialize=20)
model.eta = pyo.Var(bounds=(0, 1), initialize=0.1)

# effluent flows
components = ['A','B','C','E','P','G']

model.Feff = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':5,'G':1})

model.Feff_sum = pyo.Var(within=pyo.PositiveReals, initialize=52)

# purge, recycle, product, waste
model.FP = pyo.Var(bounds=(0, 4.763), initialize=0.5)
model.FG = pyo.Var(within=pyo.NonNegativeReals, initialize=1)
model.Fpurge = pyo.Var(within=pyo.NonNegativeReals, initialize=0)

model.FR = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':1,'B':1,'C':1,'E':1,'P':1,'G':1})

# mole fractions
model.x = pyo.Var(components, bounds=(0, 1), initialize={'A':0.2,'B':0.2,'C':0.2,'E':0.2,'P':0.2,'G':0.2})

# rate constants
model.k_reparam = pyo.Var(reaction_number, bounds=(0, None))
model.k = pyo.Var(reaction_number, bounds=(0, None), initialize={1:6.18, 2:15.2, 3:10.2})

# reaction rates
model.r = pyo.Var(reaction_number, bounds=(0, None))

# calculate the reparameterized rate constants
def k_reparam_rule(model, i):
    if i == 1:
        return model.k_reparam[i] == math.log(model.a[i]) - 120 * model.T2
    elif i == 2:
        return model.k_reparam[i] == math.log(model.a[i]) - 150 * model.T2
    else:
        return model.k_reparam[i] == math.log(model.a[i]) - 200 * model.T2
        
model.k_reparam_eq = pyo.Constraint(
    reaction_number, rule=k_reparam_rule
)

# calculate the original rate constants
def k_rule(model, i):
    return model.k[i] == pyo.exp(model.k_reparam[i])

model.k_eq = pyo.Constraint(
    reaction_number, rule=k_rule
)

# define the rates of reactions
def reaction_rate_rule(model, i):
    if i == 1:
        return model.r[i] == model.k[i] * model.x['A'] * model.x['B'] * model.V * rho
    elif i == 2:
        return model.r[i] == model.k[i] * model.x['C'] * model.x['B'] * model.V * rho
    else:
        return model.r[i] == model.k[i] * model.x['P'] * model.x['C'] * model.V * rho


model.reaction_rate = pyo.Constraint(
    reaction_number, rule=reaction_rate_rule
)

# add the temperature constraints
model.temp_rule = pyo.Constraint(
    expr = model.T2 * model.T == 1
)

# define the species mass balances
model.A_bal = pyo.Constraint(
    expr = model.Feff['A'] == model.FA + model.FR['A'] - model.r[1]
)

model.B_bal = pyo.Constraint(
    expr = model.Feff['B'] == model.FB + model.FR['B'] - (model.r[1] + model.r[2])
)

model.C_bal = pyo.Constraint(
    expr = model.Feff['C'] == model.FR['C'] + 2 * model.r[1] - 2 * model.r[2] - model.r[3]
)

model.E_bal = pyo.Constraint(
    expr = model.Feff['E'] == model.FR['E'] + 2 * model.r[2]
)

model.P_bal = pyo.Constraint(
    expr = model.Feff['P'] == 0.1 * model.FR['E'] + model.r[2] - 0.5 * model.r[3]
)

model.G_bal = pyo.Constraint(
    expr = model.Feff['G'] == 1.5 * model.r[3]
)

# calculate the total effluent flow
model.total_effluent = pyo.Constraint(
    expr = model.Feff_sum == sum(model.Feff[j] for j in components)
)

# add the constraints on the total effluent flow
model.total_effluent_constraint = pyo.Constraint(
    expr = model.Feff_sum >= 1
)

# ensure mole fraction balance
def mole_frac_rule(model, j):
    return model.Feff[j] == model.Feff_sum * model.x[j]

model.mole_frac = pyo.Constraint(components, rule=mole_frac_rule)

# calculate the product flowrate 
model.product = pyo.Constraint(
    expr = model.FP == model.Feff['P'] - 0.1 * model.Feff['E']
)

# calculate the flowrate of the waste
model.waste = pyo.Constraint(
    expr = model.FG == model.Feff['G']
)

# calculate the flowrate of the purge stream
model.purge = pyo.Constraint(
    expr = model.Fpurge == model.eta * (model.Feff['A'] + model.Feff['B'] + model.Feff['C'] + 1.1 * model.Feff['E'])
)

# calculate the flowrate of the recycle stream
def recycle_rule(model, j):
    return model.FR[j] == (1 - model.eta) * model.Feff[j]

model.recycle = pyo.Constraint(components, rule=recycle_rule)

# define the objective function
model.obj = pyo.Objective(
    expr = 100 * (2207*model.FP + 50*model.Fpurge - 168*model.FA - 252*model.FB - 2.22*model.Feff_sum 
                  - 84*model.FG - 60*model.V*rho)/(600 * model.V * rho),
    sense=pyo.maximize
)

# solve the model
solver = pyo.SolverFactory('ipopt')

# get the solved results
results = solver.solve(model, tee=True)

# print the results
print("ROI =", pyo.value(model.obj))

for v in model.component_objects(pyo.Var, active=True):
    print("\n",v)
    for index in v:
        print(index, pyo.value(v[index]))

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

# Data Generation

In [3]:
def williams_otto_model(FA, FB, temp):
    """
    Solves the Williams-Otto process model to obtain the concentrations of the products

    Parameters
    ----------
    FA: mass flowrate of pure component A
    FB: mass flowrate of pure component B
    temp: temperature of the reaction

    Returns
    -------
    Feff['P']: mass flowrate of the main product in the effluent of the reactor
    Feff['E']: mass flowrate of the by-product in the effluent of the reactor
    Feff['G']: mass flowrate of the waste product in the effluent of the reactor
    """
    model = pyo.ConcreteModel()

    # define the model parameters
    a1 = 5.9755 * 10**9
    a2 = 2.5962 * 10**12
    a3 = 9.6283 * 10**15
    reaction_number = [1, 2, 3]
    model.a = pyo.Param(reaction_number, initialize={1:a1, 2:a2, 3:a3})
    model.rho = pyo.Param(within=pyo.PositiveReals, initialize=50)
    model.E1 = pyo.Param(within=pyo.PositiveReals, initialize=120)
    model.E2 = pyo.Param(within=pyo.PositiveReals, initialize=150)
    model.E3 = pyo.Param(within=pyo.PositiveReals, initialize=200)
    
    # define the model variables
    # decision variables
    model.V = pyo.Var(bounds=(0.03, 0.1),)
    model.V.fix(0.0305)
    model.T = pyo.Var(bounds=(5.8, 6.8),)
    model.T.fix(temp)
    model.FA = pyo.Var(bounds=(1, None),)
    model.FA.fix(FA)
    model.FB = pyo.Var(bounds=(1, None),)
    model.FB.fix(FB)
    model.eta = pyo.Var(bounds=(0, 1),)
    model.eta.fix(0.1002)
    model.T2 = pyo.Var(bounds=(0.147, 1.725), initialize=0.5)
    
    # effluent flows
    components = ['A','B','C','E','P','G']
    
    model.Feff = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':5,'G':1})
    
    model.Feff_sum = pyo.Var(within=pyo.PositiveReals, initialize=52)
    
    # mole fractions
    model.x = pyo.Var(components, bounds=(0, 1), initialize={'A':0.2,'B':0.2,'C':0.2,'E':0.2,'P':0.2,'G':0.2})
    
    # rate constants
    model.k_reparam = pyo.Var(reaction_number, bounds=(0, None), initialize=1)
    model.k = pyo.Var(reaction_number, bounds=(0, None), initialize={1:6.18, 2:15.2, 3:10.2})

    # reaction rates
    model.r = pyo.Var(reaction_number, bounds=(0, None), initialize=1)
    
    # calculate the reparameterized rate constants
    def k_reparam_rule(model, i):
        if i == 1:
            return model.k_reparam[i] == math.log(model.a[i]) - model.E1 * model.T2
        elif i == 2:
            return model.k_reparam[i] == math.log(model.a[i]) - model.E2 * model.T2
        else:
            return model.k_reparam[i] == math.log(model.a[i]) - model.E3 * model.T2
            
    model.k_reparam_eq = pyo.Constraint(
        reaction_number, rule=k_reparam_rule
    )
    
    # calculate the original rate constants
    def k_rule(model, i):
        return model.k[i] == pyo.exp(model.k_reparam[i])
    
    model.k_eq = pyo.Constraint(
        reaction_number, rule=k_rule
    )
    
    # define the rates of reactions
    def reaction_rate_rule(model, i):
        if i == 1:
            return model.r[i] == model.k[i] * model.x['A'] * model.x['B'] * model.V * model.rho
        elif i == 2:
            return model.r[i] == model.k[i] * model.x['C'] * model.x['B'] * model.V * model.rho
        else:
            return model.r[i] == model.k[i] * model.x['P'] * model.x['C'] * model.V * model.rho
    
    
    model.reaction_rate = pyo.Constraint(
        reaction_number, rule=reaction_rate_rule
    )
    
    # add the temperature constraints
    model.temp_rule = pyo.Constraint(
        expr = model.T2 * model.T == 1
    )

    # define the species mass balances
    model.A_bal = pyo.Constraint(
        expr = model.Feff['A'] == model.FA - model.r[1]
    )
    
    model.B_bal = pyo.Constraint(
        expr = model.Feff['B'] == model.FB - (model.r[1] + model.r[2])
    )
    
    model.C_bal = pyo.Constraint(
        expr = model.Feff['C'] == 2 * model.r[1] - 2 * model.r[2] - model.r[3]
    )
    
    model.E_bal = pyo.Constraint(
        expr = model.Feff['E'] == 2 * model.r[2]
    )
    
    model.P_bal = pyo.Constraint(
        expr = model.Feff['P'] == model.r[2] - 0.5 * model.r[3]
    )
    
    model.G_bal = pyo.Constraint(
        expr = model.Feff['G'] == 1.5 * model.r[3]
    )
    
    # calculate the total effluent flow
    model.total_effluent = pyo.Constraint(
        expr = model.Feff_sum == sum(model.Feff[j] for j in components)
    )
    
    # add the constraints on the total effluent flow
    model.total_effluent_constraint = pyo.Constraint(
        expr = model.Feff_sum >= 1
    )
    
    # ensure mole fraction balance
    def mole_frac_rule(model, j):
        return model.Feff[j] == model.Feff_sum * model.x[j]
    
    model.mole_frac = pyo.Constraint(components, rule=mole_frac_rule)
    
    # solve the model
    solver = pyo.SolverFactory('ipopt')
    
    # get the solved results
    results = solver.solve(model, tee=True)
    
    for v in model.component_objects(pyo.Var, active=True):
        print("\n",v)
        for index in v:
            print(index, pyo.value(v[index]))

    products_mass_flow = [pyo.value(model.Feff['P']), pyo.value(model.Feff['G']), pyo.value(model.Feff['E'])]
    print("The flowrate of the products is:", {'P':products_mass_flow[0], 'G':products_mass_flow[1], 'E':products_mass_flow[2]})

    return products_mass_flow

## Fullfactorial Model Simulation

In [4]:
# flowrate and temperature conditions
FA_conditions = [10, 15]
FB_conditions = [25, 35]
temp_conditions = [5.8, 6.8]

# factor levels
levels = [2, 2, 2]

# generating design matrix
design_matrix = fullfact(levels)

# name of factors
factor_names = ["FA", "FB", "temp"]

# design matrix dataframe
df = pd.DataFrame(design_matrix, columns=factor_names)
print(df)

# factors combination corresponding to design matrix
inputs = []
outputs = []

# function that scales design matrix values to factor values
def rescale(x, upper_bounds, lower_bounds):
    """
    Evaluates corresponding factor value of design matrix value

    Parameters
    ----------
    x: design matrix value
    upper_bounds: factor high level
    lower_bounds: factor low level

    Returns
    -------
    new_x: Corresponding factor value"""
    new_x = x*(upper_bounds - lower_bounds) + lower_bounds
    return(new_x)

# looping through the rows of design matrix dataframe
for i in range(len(design_matrix)):
    exp_vals = df.iloc[i]

    # scale design matrix values to factor values
    FA = rescale(exp_vals["FA"], FA_conditions[1], FA_conditions[0])
    FB = rescale(exp_vals["FB"], FB_conditions[1], FB_conditions[0])
    temp = rescale(exp_vals["temp"], temp_conditions[1], temp_conditions[0])

    inputs.append([FA, FB, temp])
    
    # Perform experiments
    new_outputs = williams_otto_model(FA, FB, temp)
    outputs.append(new_outputs)

    FA   FB  temp
0  0.0  0.0   0.0
1  1.0  0.0   0.0
2  0.0  1.0   0.0
3  1.0  1.0   0.0
4  0.0  0.0   1.0
5  1.0  0.0   1.0
6  0.0  1.0   1.0
7  1.0  1.0   1.0
Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes w

In [5]:
# name of experiment inputs
input_names = ["FA", "FB", "temp"]

# name of experiment outputs
output_names = ["FP", "FG", "FE"]

# inputs dataframe
inputs_df = pd.DataFrame(inputs[0:4], columns=input_names)

# outputs dataframe
outputs_df = pd.DataFrame(outputs[0:4], columns=output_names)

print(inputs_df)
print(outputs_df)

     FA    FB  temp
0  10.0  25.0   5.8
1  15.0  25.0   5.8
2  10.0  35.0   5.8
3  15.0  35.0   5.8
         FP        FG        FE
0  0.682486  0.020686  1.378763
1  0.685827  0.021142  1.385748
2  0.571036  0.009943  1.148700
3  0.617530  0.011762  1.242902


## Add Gaussian Noise

In [6]:
# get the flowrate of P
FP_prdt_flow = outputs_df["FP"]

# standard deviation
P_E_flow_std = 0.01

# creating gaussian noise
np.random.seed(10)
FP_noise = stats.norm.rvs(loc=0, scale=P_E_flow_std, size=len(FP_prdt_flow))

# corrupt the flowrate with the noise
noisy_FP_prdt_flow = FP_prdt_flow + FP_noise

In [7]:
# get the flowrate of G
FG_prdt_flow = outputs_df["FG"]

# standard deviation
G_flow_std = 0.001

# creating gaussian noise
np.random.seed(10)
FG_noise = stats.norm.rvs(loc=0, scale=G_flow_std, size=len(FG_prdt_flow))

# corrupt the flowrate with the noise
noisy_FG_prdt_flow = FG_prdt_flow + FG_noise

In [8]:
# get the flowrate of E
FE_prdt_flow = outputs_df["FE"]

# creating gaussian noise
np.random.seed(10)
FE_noise = stats.norm.rvs(loc=0, scale=P_E_flow_std, size=len(FE_prdt_flow))

# corrupt the flowrate with the noise
noisy_FE_prdt_flow = FE_prdt_flow + FE_noise

In [9]:
# inputs data
FA_feed = inputs_df["FA"]
FB_feed = inputs_df["FB"]
temp_reactor = inputs_df["temp"]

# data columns
column_names = ["FA", "FB", "temp", "FP", "FG", "FE"]

# number of data columns
n_columns = len(column_names)

# number of data points
n_data = len(FA_feed)

data_matrix = np.zeros((n_data, n_columns))
data_matrix[:, 0] = FA_feed
data_matrix[:, 1] = FB_feed
data_matrix[:, 2] = temp_reactor
data_matrix[:, 3] = noisy_FP_prdt_flow
data_matrix[:, 4] = noisy_FG_prdt_flow
data_matrix[:, 5] = noisy_FE_prdt_flow

# full data
data = pd.DataFrame(data_matrix, columns=column_names)
print(data)

williams_otto_data = data.to_csv("williams_otto_data.csv", index=False)

     FA    FB  temp        FP        FG        FE
0  10.0  25.0   5.8  0.695802  0.022017  1.392079
1  15.0  25.0   5.8  0.692980  0.021857  1.392901
2  10.0  35.0   5.8  0.555582  0.008398  1.133246
3  15.0  35.0   5.8  0.617446  0.011754  1.242818
